# LLM from Scratch - Training Notebook

This notebook builds and trains a small chat LLM (~50M params) from scratch.

**Instructions:**
1. Go to Runtime > Change runtime type > Select **T4 GPU**
2. Run all cells sequentially (Runtime > Run all)
3. The final cell runs an interactive chat loop

In [ ]:
# Cell 1: Install dependencies
!pip install tiktoken datasets -q
print('Dependencies installed!')

In [ ]:
# Cell 2: Import everything
import math, time, json, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
import tiktoken
from datasets import load_dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Cell 3: Model Architecture

def precompute_rope_freqs(dim, max_seq_len, theta=10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(max_seq_len).float()
    freqs = torch.outer(t, freqs)
    return torch.polar(torch.ones_like(freqs), freqs)

def apply_rope(x, freqs_cis):
    x_complex = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
    freqs_cis = freqs_cis.to(x_complex.device)
    x_rotated = torch.view_as_real(x_complex * freqs_cis).flatten(-2)
    return x_rotated.type_as(x)

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        norm = x.float().pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return (x.float() * norm).type_as(x) * self.weight

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len, bias=False):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.wq = nn.Linear(d_model, d_model, bias=bias)
        self.wk = nn.Linear(d_model, d_model, bias=bias)
        self.wv = nn.Linear(d_model, d_model, bias=bias)
        self.wo = nn.Linear(d_model, d_model, bias=bias)
        self.register_buffer('freqs_cis', precompute_rope_freqs(self.head_dim, max_seq_len), persistent=False)
        self.scale = self.head_dim ** -0.5
    def forward(self, x, mask):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        q = apply_rope(q, self.freqs_cis[:T])
        k = apply_rope(k, self.freqs_cis[:T])
        scores = (q @ k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(mask[:, :, :T, :T] == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.wo(out)

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False)
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads, max_seq_len)
        self.ffn = SwiGLU(d_model, d_ff)
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
    def forward(self, x, mask):
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x

class TransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_layers=12, n_heads=8, d_ff=1376, max_seq_len=2048):
        super().__init__()
        self.config = {'vocab_size': vocab_size, 'd_model': d_model, 'n_layers': n_layers,
                       'n_heads': n_heads, 'd_ff': d_ff, 'max_seq_len': max_seq_len}
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([TransformerBlock(d_model, n_heads, d_ff, max_seq_len) for _ in range(n_layers)])
        self.norm = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.apply(self._init_weights)
        n_params = sum(p.numel() for p in self.parameters())
        print(f'Model: {n_params/1e6:.1f}M parameters')

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, std=0.02)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        mask = torch.tril(torch.ones(T, T, device=idx.device)).unsqueeze(0).unsqueeze(0)
        x = self.tok_emb(idx)
        for layer in self.layers:
            x = layer(x, mask)
        x = self.norm(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        return logits, loss

print('Architecture defined!')

In [ ]:
# Cell 4: Tokenizer and Dataset classes

enc = tiktoken.get_encoding('gpt2')
VOCAB_SIZE = enc.n_vocab
print(f'Vocabulary size: {VOCAB_SIZE}')

class PretrainDataset(Dataset):
    def __init__(self, tokens, seq_len=512):
        self.seq_len = seq_len
        self.examples = []
        for i in range(0, len(tokens) - seq_len - 1, seq_len):
            self.examples.append((tokens[i:i+seq_len], tokens[i+1:i+1+seq_len]))
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        x, y = self.examples[idx]
        return {'input_ids': torch.tensor(x, dtype=torch.long), 'targets': torch.tensor(y, dtype=torch.long)}

class ChatDataset(Dataset):
    def __init__(self, conversations, max_len=1024):
        self.examples = []
        for conv in conversations:
            tokens = []
            for turn in conv:
                role = turn.get('role', 'user')
                content = turn.get('content', '')
                if role == 'user':
                    tokens.extend(enc.encode(f'<|user|>\n{content}\n'))
                elif role == 'assistant':
                    tokens.extend(enc.encode(f'<|assistant|>\n{content}\n<|end|>\n'))
            for i in range(0, max(1, len(tokens) - max_len), max_len // 2):
                chunk = tokens[i:i+max_len]
                if len(chunk) > 1:
                    self.examples.append({
                        'input_ids': torch.tensor(chunk[:-1], dtype=torch.long),
                        'targets': torch.tensor(chunk[1:], dtype=torch.long),
                    })
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        return self.examples[idx]

print('Dataset classes defined!')

In [ ]:
# Cell 5: Training utilities

def get_optimizer(model, lr=3e-4, weight_decay=0.1):
    return AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=weight_decay)

def get_scheduler(optimizer, warmup_steps, total_steps):
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return LambdaLR(optimizer, lr_lambda)

def train_model(model, optimizer, scheduler, train_loader, val_loader=None,
                num_epochs=1, grad_accum=4, log_every=10, eval_every=500, device='cuda'):
    model.to(device)
    use_amp = device == 'cuda'
    scaler = torch.amp.GradScaler(device) if use_amp else None
    best_val_loss = float('inf')

    for epoch in range(1, num_epochs + 1):
        print(f'\n--- Epoch {epoch}/{num_epochs} ---')
        model.train()
        total_loss = 0
        t0 = time.time()

        for step, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            targets = batch['targets'].to(device)

            ctx = torch.amp.autocast(device_type=device, dtype=torch.bfloat16) if use_amp else torch.no_grad().__class__()
            with ctx if use_amp else torch.enable_grad():
                _, loss = model(input_ids, targets)
                loss = loss / grad_accum

            if scaler:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            total_loss += loss.item() * grad_accum

            if (step + 1) % grad_accum == 0:
                if scaler:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            if (step + 1) % log_every == 0:
                avg = total_loss / (step + 1)
                elapsed = time.time() - t0
                tok_s = (log_every * train_loader.batch_size * input_ids.shape[1]) / elapsed
                lr_now = scheduler.get_last_lr()[0]
                print(f'Step {step+1}/{len(train_loader)} | Loss: {avg:.4f} | LR: {lr_now:.2e} | {tok_s:.0f} tok/s')
                t0 = time.time()

            if (step + 1) % eval_every == 0 and val_loader:
                val_loss = evaluate(model, val_loader, device)
                print(f'  Val Loss: {val_loss:.4f} | PPL: {math.exp(val_loss):.2f}')
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    torch.save(model.state_dict(), 'best_model.pt')
                    print('  Saved best model!')

    return best_val_loss

def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    n = 0
    with torch.no_grad():
        for batch in loader:
            _, loss = model(batch['input_ids'].to(device), batch['targets'].to(device))
            total_loss += loss.item()
            n += 1
    model.train()
    return total_loss / max(n, 1)

print('Training utilities defined!')

In [ ]:
# Cell 6: Load and tokenize pre-training data
SEQ_LEN = 512
BATCH_SIZE = 2

print('Loading Alpaca dataset for pre-training text...')
ds = load_dataset('tatsu-lab/alpaca', split='train', streaming=True)

print('Tokenizing first 5k examples...')
all_tokens = []
for i, example in enumerate(ds):
    if i >= 5000:
        break
    text = example.get('instruction', '') + '\n' + example.get('input', '') + '\n' + example.get('output', '')
    all_tokens.extend(enc.encode(text))
    if i % 5000 == 0:
        print(f'  Processed {i} examples, {len(all_tokens):,} tokens')

print(f'Total tokens: {len(all_tokens):,}')

dataset = PretrainDataset(all_tokens, SEQ_LEN)
val_size = min(len(dataset) // 20, 1000)
train_size = len(dataset) - val_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train: {len(train_dataset)} samples | Val: {len(val_dataset)} samples')

In [ ]:
# Cell 7: Build model and pre-train
model = TransformerLM(
    vocab_size=VOCAB_SIZE,
    d_model=256,
    n_layers=4,
    n_heads=4,
    d_ff=640,
    max_seq_len=SEQ_LEN,
)

optimizer = get_optimizer(model, lr=3e-4)
total_steps = len(train_loader) * 3  # 3 epochs
scheduler = get_scheduler(optimizer, warmup_steps=100, total_steps=total_steps)

print('\nStarting pre-training...')
best = train_model(model, optimizer, scheduler, train_loader, val_loader,
                   num_epochs=3, grad_accum=4, log_every=10, eval_every=500, device=device)

torch.save(model.state_dict(), 'pretrained.pt')
with open('pretrained_config.json', 'w') as f:
    json.dump(model.config, f)
print(f'\nPre-training done! Best val loss: {best:.4f}')

In [ ]:
# Cell 8: Load chat dataset and fine-tune
print('Loading Guanaco chat dataset...')
chat_ds = load_dataset('mlabonne/guanaco-llama2-1k', split='train', streaming=True)

conversations = []
for i, example in enumerate(chat_ds):
    if i >= 10000:
        break
    text = example.get('text', '')
    conv = []
    parts = text.split('### ')
    for part in parts:
        part = part.strip()
        if part.startswith('Human:'):
            conv.append({'role': 'user', 'content': part[len('Human:'):].strip()})
        elif part.startswith('Assistant:'):
            conv.append({'role': 'assistant', 'content': part[len('Assistant:'):].strip()})
    if conv:
        conversations.append(conv)

print(f'Loaded {len(conversations)} conversations')

chat_dataset = ChatDataset(conversations, max_len=SEQ_LEN)
val_size = min(len(chat_dataset) // 20, 500)
train_size = len(chat_dataset) - val_size
chat_train, chat_val = random_split(chat_dataset, [train_size, val_size])

chat_train_loader = DataLoader(chat_train, batch_size=2, shuffle=True, num_workers=0, pin_memory=True)
chat_val_loader = DataLoader(chat_val, batch_size=2, shuffle=False, num_workers=0)

print(f'Chat train: {len(chat_train)} | Chat val: {len(chat_val)}')

# Load pre-trained weights
model.load_state_dict(torch.load('pretrained.pt', map_location=device, weights_only=True))
print('Loaded pre-trained weights')

# Fine-tune with lower LR
optimizer = get_optimizer(model, lr=2e-5)
total_steps = len(chat_train_loader) * 3
scheduler = get_scheduler(optimizer, warmup_steps=50, total_steps=total_steps)

print('\nStarting SFT training...')
best = train_model(model, optimizer, scheduler, chat_train_loader, chat_val_loader,
                   num_epochs=3, grad_accum=8, log_every=10, eval_every=500, device=device)

torch.save(model.state_dict(), 'chat_model.pt')
with open('chat_model_config.json', 'w') as f:
    json.dump(model.config, f)
print(f'\nSFT done! Best val loss: {best:.4f}')

In [ ]:
# Cell 9: Text generation

@torch.no_grad()
def generate(prompt, max_new_tokens=256, temperature=0.8, top_k=50, top_p=0.9):
    model.eval()
    tokens = enc.encode(prompt)
    x = torch.tensor([tokens], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        x_cond = x if x.size(1) <= model.config['max_seq_len'] else x[:, -model.config['max_seq_len']:]
        logits, _ = model(x_cond)
        logits = logits[:, -1, :] / temperature

        if top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')

        if top_p < 1.0:
            sorted_logits, sorted_idx = torch.sort(logits, descending=True)
            cum = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
            remove = cum > top_p
            remove[..., 1:] = remove[..., :-1].clone()
            remove[..., 0] = 0
            sorted_logits[remove] = float('-inf')
            logits = sorted_logits.scatter(1, sorted_idx, sorted_logits)

        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, 1)
        x = torch.cat([x, next_token], dim=1)

    return enc.decode(x[0].tolist())

# Test it
print(generate('The meaning of life is', max_new_tokens=100))

In [ ]:
# Cell 10: Interactive chat
print('Chat with your LLM! Type quit to exit.\n')

while True:
    user_input = input('You: ').strip()
    if user_input.lower() in ('quit', 'exit', 'q'):
        break
    if not user_input:
        continue

    prompt = f'<|user|>\n{user_input}\n<|assistant|>\n'
    response = generate(prompt, max_new_tokens=256, temperature=0.7)
    assistant_reply = response.split('<|assistant|>\n')[-1].split('<|end|>')[0].strip()
    print(f'Assistant: {assistant_reply}\n')

In [ ]:
# Cell 11: Download your model (optional)
from google.colab import files

try:
    files.download('chat_model.pt')
    files.download('chat_model_config.json')
    print('Downloaded!')
except:
    print('No files to download or not running in Colab')

# Also save to Google Drive if mounted
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    shutil.copy('chat_model.pt', '/content/drive/MyDrive/chat_model.pt')
    shutil.copy('chat_model_config.json', '/content/drive/MyDrive/chat_model_config.json')
    print('Saved to Google Drive!')
except:
    print('Google Drive not mounted')